# 01_09 — Compute & save isoform fractions

In [ ]:
print("Hello World")

In [1]:
import logging
logging.getLogger("fontTools").setLevel(logging.WARNING)

import os
_r = os.path.abspath(".")
while _r != os.path.dirname(_r) and not os.path.exists(os.path.join(_r, ".notebooks_root")):
    _r = os.path.dirname(_r)
os.chdir(_r if os.path.exists(os.path.join(_r, ".notebooks_root")) else "/oak/stanford/groups/quake/mmantri/group.quake/tabula_longread/notebooks")  # cd to notebooks/ root (marked .notebooks_root); relocation-proof

import os, sys, gc, logging
import numpy as np
import scanpy as sc

sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))
from isoform_fraction import compute_isoform_fractions

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
gc.enable()

BASEDIR = "./../pacbio"
H5AD_DIR = f"{BASEDIR}/h5ads"

LEVELS = ["pbids", "ensemblids", "ensemblids_annotatedonly"]
MIN_CELLS_PER_ISOFORM = 1   # keep isoforms detected in >= this many cells
MIN_ISOFORMS_PER_GENE = 2    # keep genes with >= this many isoforms
N_THREADS = 16

In [ ]:
def build_isofrac(level, save=True, n_threads=N_THREADS):
    in_path = f"{H5AD_DIR}/all_samples_pacbio_recollapsed_raw_counts_{level}_bc_anndata_preprocessed.h5ad"
    adata = sc.read_h5ad(in_path)
    n0 = adata.shape

    # 1) isoforms detected in >= MIN_CELLS_PER_ISOFORM cells. This is 1, i.e.
    #    deliberately a NO-OP: 01_08 already required >= 5 cells. Kept as a
    #    hook. The only effective filter is step 2.
    cell_counts = np.asarray((adata.X > 0).sum(axis=0)).ravel()
    adata = adata[:, cell_counts >= MIN_CELLS_PER_ISOFORM].copy()

    # 2) genes with >= 2 isoforms (single-isoform genes have a trivial fraction of 1)
    gene_col = "associated_gene" if "associated_gene" in adata.var.columns else "gene_name"
    gene_counts = adata.var[gene_col].value_counts()
    multi_iso_genes = gene_counts[gene_counts >= MIN_ISOFORMS_PER_GENE].index
    adata = adata[:, adata.var[gene_col].isin(multi_iso_genes)].copy()
    gc.collect()

    # 2b) RECOMPUTE QC. obs stats arrive from 01_08 computed on the PRE-SUBSET
    #     feature set; step 2 just dropped features, so obs['total_counts']
    #     would otherwise be a library size over features this object no
    #     longer contains and would disagree with adata.X.sum(axis=1).
    #     var stats are already valid (per-feature), so this is a no-op there.
    sc.pp.calculate_qc_metrics(adata, inplace=True)

    # 3) isoform-fraction layer
    compute_isoform_fractions(adata, gene_col=gene_col, n_threads=n_threads)

    out_path = f"{H5AD_DIR}/all_samples_pacbio_recollapsed_raw_counts_{level}_bc_anndata_preprocessed_with_isofrac.h5ad"
    if save:
        adata.write_h5ad(out_path)

    print(f"[{level}] {n0} -> {adata.shape} "
          f"(>= {MIN_CELLS_PER_ISOFORM} cells, >= {MIN_ISOFORMS_PER_GENE} iso/gene, gene_col={gene_col}) "
          f"| isoform_fraction nnz={adata.layers['isoform_fraction'].nnz:,} | saved {out_path}")
    del adata
    gc.collect()


for lvl in LEVELS:
    build_isofrac(lvl)

2026-09-01 01:05:13,184 INFO Computing isoform fractions: 203311 cells x 847094 isoforms
2026-09-01 01:05:17,598 INFO Grouped into 30489 genes (threads=16, batch=500)
